# Su and Judd (2012)
The goal of this project is to implement the estimation procedure laid out in Su and Judd (2012). I provide a brief overview of the setting and their estimation procedure below, but encourage readers to read the full paper. 

The key result from Su and Judd is that **we can write maximum likelihood problems as constrained optimization problems**, and avoid computation of a fixed point equation. They call this class of estimation Mathematical Program with Equilibrium Constraints (MPEC). 

As is standard in the literature, the authors use the setting of Rust (1987) to motivate their estimation. If you are unfamiliar with Rust (which is unlikely if you are reading this), I recommend reading my notes available in this repository, as well as the original paper. 

## Setting

 The infamous Harold Zurcher returns yet again, and must decide on more bus engine replacements. His replacement problems consists of a state $s_t$ that is the mileage of a given bus and an action $a_t$ at time $t$. The action is whether to replace the bus engine $(a_t=1)$ or not $(a_t=0)$. 

 I now summarize the problem, Rust's solution, develop necessary notation, and move into the MPEC approach. 

### Bus maintenance
HZ observes mileage $s_t$ at time $t$, and chooses between engine replacement $a_t=1$ or not $a_t=0$, where not replacing the engine still requires ordinary maintenance:

$$
u(s_t, a_t;\theta) = \begin{cases}
-c(s_t;\theta) & \text{ if } a_t=0 \\ 
-(RC + c(0; \theta)) & \text { if } a_t=1
\end{cases}
$$

You will often see this represented as (and is the functional form I estimate below) 
$$
u(s_t, a_t, \varepsilon_{0t}, \varepsilon_{1t};\theta) = \begin{cases}
-\theta_1 s_t -\theta_2 s_t^2 + \varepsilon_{0t} & \text{ if } a_t = 0 \\ 
-\theta_3 + \varepsilon_{1t} & \text{ if } a_t =1 
\end{cases}
$$

where $\theta = (\theta_1, \theta_2, \theta_3)'$.  $\theta_1$ and $\theta_2$ therefore represent maintenance costs given mileage $s_t$, and $\theta_3$ is the fixed cost of replacing the engine. 

It is important to highlight that what we ultimately want to estimate are these parameters $\theta$. 

HZ solves the dynamic programming problem 

$$
V_\theta(s_t, \varepsilon_t)= \max_{ \{a_\tau\}_{\tau=0}^\infty } \mathbb{E}\Big[ {\sum_{\tau=t}^\infty} \beta^{\tau-t}u(s_\tau, a_\tau, \varepsilon_\tau;\theta) \Big| s_t, \varepsilon_t \Big]
$$

where $\varepsilon_t=(\varepsilon_{0t}, \varepsilon_{1t})$. We could also write this as a Bellman, which is straightforward thus omitted here. 

As the econometrician, we observe the mileage $x$ and action $a$ in each period, but not cost. That is, we observe

$$
\{s_t, a_t \}_{t=1}^T
$$

We assume that the errors follow a type I extreme value distribution. 

In addition to the parameters $\theta$, we also want to estimate the transition probabilities $p(s_{t+1}|s_t, a_t;\theta)$.

Now consider the likelihood function: 

$$
L(\theta)=\Pi_{t=2}^T P(a_t|s_t;\theta)p(s_t|s_{t-1}, a_{t-1};\theta)
$$

where 

$$
P(a|s;\theta)=\frac{\exp(u(s,a;\theta)+\beta EV_\theta(s,a))}{\sum_{a'\in\{0,1 \}} \exp(u(s,a';\theta)+\beta EV_{\theta}(s,a'))}
$$

$$
EV_\theta(s,a)
= T_\theta(EV_\theta)(s,a) \equiv
\int
\log\left(
\sum_{a'\in\{0,1\}}
\exp\left(
u(s',a';\theta)
+
\beta EV_\theta(s',a')
\right)
\right)
\,p(ds' \mid s,a;\theta).
$$

To quickly summarize the solution, the outer loop solves the likelihood function, and the inner loop computes the expected value function for a given $\theta$ by solving 

$$
EV_\theta=T_\theta(EV_\theta)
$$

which is a nested fixed point problem. We have to compute $EV_\theta$ to high accuracy for each $\theta$ examined for

1. the outer loop to converge and 
2. to obtain accurate numerical derivatives for the outer loop. 

### Return to Su and Judd (2012)
The estimation of HZ's decision problem in Rust relies upon Value Function Iteration (VFI), which is computationally expensive. 

Su and Judd suggest forming an augmented likelihood function for the data 

$$
X=
\{s_t, a_t \}_{t=1}^T

$$

$$
\mathcal{L}(\theta, EV;X)=\Pi_{t=2}^T P(a_t | s_t; \theta)p(s_t|s_{t-1}, a_{t-1};\theta)
$$

$$
P(a|s;\theta)=\frac{\exp(u(s,a;\theta)+\beta EV_\theta(s,a))}{\sum_{a'\in\{0,1 \}} \exp(u(s,a';\theta)+\beta EV_{\theta}(s,a'))}
$$

 as before. From here, the authors show that rationality and the Bellman equation impose a relationship between $\theta$ and $EV$: 

$$
EV=T(EV,\theta)
$$

 Using this, we can solve the constrained optimaization problem

 $$
\max_{\theta, EV} \quad  \mathcal{L}(\theta, EV;X) \\ 
\quad \text{ subject to } \quad EV=T(EV,\theta)
 $$

## Estimation
I now turn to the estimation algorithm of Su and Judd. For simplicity, I load a simulated dataset based on the Rust DGP -- code for its generation is found in this repo in `Rust.ipynb`. I also lift the discount factor $\beta$ from the same DGP. 

### Setup

In [ ]:
import numpy as np
import pandas as pd

from scipy.optimize import minimize, NonlinearConstraint
from scipy.special import expit



df = pd.read_csv('../rust.csv')
df.head()

# Set parameters from the original DGP used for HZ's problem 
theta = [0.13, -0.004, 3.1] # true parameters 
beta = 0.95 # discount factor 
gamma = np.euler_gamma # Euler's constant 
k=10 # mileage bins
s=np.arange(k)
lam = 0.82 # transition probability

In [9]:
x, a = df['State'].values, df['Action'].values
n1 = np.bincount(x[a==1], minlength=k).astype(float)
n0 = np.bincount(x[a==0], minlength=k).astype(float)
n  = n0 + n1
assert len(n1) == k

mask    = (df['Action'] == 0) & (df['State'] < df['State'].max())
est_lam = (df.loc[mask, 'State1'] - df.loc[mask, 'State']).mean()
T0 = np.zeros((k, k))
for xx in range(k):
    T0[xx, xx]             += 1 - est_lam
    T0[xx, min(xx+1, k-1)] += est_lam
r = T0[0] 

We've now handled all the prerequisites. We can now turn to using MPEC for estimation of HZ's problem.

In [14]:
def split(p): return p[0], p[1], p[2], p[3:]

def vfuncs(p): # choice/alternative-specific value functions
    t1, t2, t3, V = split(p); Ek = T0 @ V
    v0 = -t1*s - t2*s**2 + beta*Ek           # c(s,a=0;theta) = -\theta_1 s -\theta_2 s^2 + eps 
    v1 = -t3            + beta*Ek[0]          # replace: -θ3, continue from state 0
    return v0, v1

def nll(p): # (negative) log likelhihood
    v0, v1 = vfuncs(p); p1 = expit(v1 - v0)
    return -(n1*np.log(p1+1e-15) + n0*np.log(1-p1+1e-15)).sum()

def nll_grad(p): # gradient of negative log likelihood
    v0, v1 = vfuncs(p); p1 = expit(v1 - v0)
    g = n*p1 - n1; G = g.sum()
    return np.concatenate(([(g*s).sum(), (g*s**2).sum(), -G], beta*(G*r - g@T0)))

def con(p):                                  # Bellman equation
    v0, v1 = vfuncs(p); _, _, _, V = split(p)
    return V - gamma - np.logaddexp(v0, v1)

def con_jac(p): # Jacobian
    v0, v1 = vfuncs(p); p1 = expit(v1 - v0)
    J = np.zeros((k, 3+k))
    J[:,0], J[:,1], J[:,2] = (1-p1)*s, (1-p1)*s**2, p1
    J[:,3:] = np.eye(k) - beta*((1-p1)[:,None]*T0 + p1[:,None]*r[None,:])
    return J

nlc = NonlinearConstraint(con, 0, 0, jac=con_jac)
p0  = np.zeros(3 + k)   # theta0 = [0,0,0], our initial guess
res = minimize(nll, p0, jac=nll_grad, method='trust-constr', constraints=[nlc], # minimize nll w/ gradient
               options={'xtol':1e-11, 'gtol':1e-11, 'maxiter':800})  
t1, t2, t3, V = split(res.x) # grab the estimated parameters

# standard errors via implicit function theorem through the Bellman constraints 
v0, v1 = vfuncs(res.x); p1 = expit(v1 - v0); J = con_jac(res.x) 
dV = -np.linalg.solve(J[:,3:], J[:,:3])
dD = np.column_stack([s, s**2, -np.ones(k)]) + beta*(r[None,:]-T0) @ dV
I  = (dD * (n*p1*(1-p1))[:,None]).T @ dD
se = np.sqrt(np.diag(np.linalg.inv(I)))

# Results
print(f"lambda_hat = {est_lam:.4f}")
for nm, est, e in zip(['theta1','theta2','theta3'], [t1,t2,t3], se):
    print(f"{nm} = {est:8.4f}  (se {e:.4f})")
print(f"True cost parameters are {theta}, and the true transition probability was {lam}.")

lambda_hat = 0.8202
theta1 =   0.1315  (se 0.0074)
theta2 =  -0.0041  (se 0.0007)
theta3 =   3.0986  (se 0.0371)
True cost parameters are [0.13, -0.004, 3.1], and the true transition probability was 0.82.


## Summary
In my implementation, Rust's (1987) NFXP ran in about 3.5 seconds (see `Rust.ipynb`), while the Su and Judd (2012) MPEC converged in a few milliseconds. Two caveats to keep in mind: 

1. NFXP used derivative-free Nelder–Mead, whereas MPEC supplied an analytic gradient and sparse constraint Jacobian, so part of the speedup reflects derivative information rather than the algorithm itself.

2. Given a small state space and a discount factor of $\beta=0.95$ both problems are computationally inexpensive. MPEC's advantage emerges as the state space grows or β approaches one — NFXP must re-solve the Bellman fixed point to high precision at every trial parameter vector, an inner loop whose cost grows with the state space, while MPEC imposes the Bellman equation as constraints and solves the problem jointly.

### Notes
This general approach can also be applied to demand estimation. Specifically, Dube et al. (2012) use the same constrained optimization approach to solve for the fixed point associated with the inversion of market shares. 

### References
Dube, J.P., Fox, J. and Su, C.-L (2012), "Improving the Numerical Performance of Static and Dynamic Aggregate Discrete Choice Random Coefficients Demand Estimation." Econometrica, 80: 2231-2267

Rust, John. 1987. “Optimal Replacement of GMC Bus Engines: An Empirical Model of Harold Zurcher.” Econometrica, 55: 999-1033

Su, C.-L. and Judd, K.L. (2012), "Constrained Optimization Approaches to Estimation of Structural Models." Econometrica, 80: 2213-2230.